# 08 – String Operations in Pandas

Text columns are messy in real data.

Names typed in different cases — `"amit"`, `"AMIT"`, `"Amit"`. Phone numbers with spaces or dashes. City names with trailing spaces. Email addresses in wrong format. Product codes mixed with descriptions.

Before you can analyse or group text data, you need to clean it.

Pandas gives you the `.str` accessor — it lets you apply string operations to an entire column in one line, without writing any loops.

You already saw `.str.startswith()` and `.str.contains()` in the filtering notebook. Here we go much deeper.

## Setting Up

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Name":    ["  amit sharma ", "PRIYA MEHTA", "ravi Kumar", "neha SINGH", "Karan das"],
    "Email":   ["amit@gmail.com", "PRIYA@yahoo.com", "ravi.kumar@gmail.com",
                "neha@outlook.com", "karan123@gmail.com"],
    "City":    ["pune", "Mumbai ", " delhi", "PUNE", "mumbai"],
    "Phone":   ["98765-43210", "876 543 2100", "9812345678", "  9900112233", "77-8899-0011"]
})

print(df)

Look at the problems:
- `Name` — inconsistent casing, extra spaces
- `Email` — inconsistent casing
- `City` — lowercase, uppercase, trailing/leading spaces, inconsistent
- `Phone` — dashes, spaces, different formats

All of this needs cleaning before any grouping or analysis.

## 1) Changing Case

The most common first step — standardise the casing so `"pune"`, `"Pune"`, and `"PUNE"` are all treated the same.

In [ ]:
# .str.lower() — everything to lowercase
print(df["City"].str.lower())

In [ ]:
# .str.upper() — everything to uppercase
print(df["Name"].str.upper())

In [ ]:
# .str.title() — Title Case (first letter of every word capitalised)
print(df["Name"].str.title())

In [ ]:
# .str.capitalize() — only the first letter capitalised, rest lowercase
print(df["Name"].str.strip().str.capitalize())

In [ ]:
# Apply and save — standardise City column
df["City"] = df["City"].str.strip().str.title()
print(df["City"])

## 2) Removing Extra Spaces with `strip()`

Leading and trailing spaces are invisible but cause real problems. `"Pune"` and `"Pune "` look the same but are not equal — so groupby and filtering will treat them as different values.

In [ ]:
# .str.strip() — removes spaces from both ends
print(df["Name"].str.strip())

In [ ]:
# .str.lstrip() — remove only from the left (start)
# .str.rstrip() — remove only from the right (end)

print(df["Phone"].str.lstrip())

In [ ]:
# Clean the Name column — strip spaces and apply title case
df["Name"] = df["Name"].str.strip().str.title()
print(df["Name"])

## 3) Replacing Characters with `replace()`

Use `.str.replace()` to swap out specific characters or patterns in a string column.

In [ ]:
# Remove dashes and spaces from phone numbers to standardise them

df["Phone"] = df["Phone"].str.strip()
df["Phone"] = df["Phone"].str.replace("-", "", regex=False)
df["Phone"] = df["Phone"].str.replace(" ", "", regex=False)

print(df["Phone"])

`regex=False` tells pandas to treat the first argument as a plain string, not a regular expression pattern. Always set this when you're replacing literal characters.

In [ ]:
# Lowercase all emails — emails should always be lowercase

df["Email"] = df["Email"].str.lower()
print(df["Email"])

## 4) Splitting Strings

`.str.split()` breaks a string into parts based on a separator character — like splitting a full name into first and last name.

In [ ]:
# Split Name into First and Last name
# expand=True puts each part into its own column

name_split = df["Name"].str.split(" ", expand=True)
print(name_split)

In [ ]:
# Assign them as new columns

df["First_Name"] = df["Name"].str.split(" ").str[0]
df["Last_Name"]  = df["Name"].str.split(" ").str[1]

print(df[["Name", "First_Name", "Last_Name"]])

In [ ]:
# Split email to extract domain name

df["Domain"] = df["Email"].str.split("@").str[1]
print(df[["Email", "Domain"]])

## 5) Extracting Parts with `str[]`

You can index into a string column just like a list — `str[0]` gives the first character, `str[:3]` gives the first 3 characters.

In [ ]:
# First character of each name

print(df["First_Name"].str[0])

In [ ]:
# First 3 characters of phone number — could be area code

print(df["Phone"].str[:3])

In [ ]:
# Last 10 characters — useful to trim if there's a country code prefix

print(df["Phone"].str[-10:])

## 6) Searching Inside Strings

These methods check if a pattern exists in each value — they return `True` or `False`, making them perfect for filtering.

In [ ]:
# Which emails are from gmail?

gmail_users = df[df["Email"].str.contains("gmail", na=False)]
print(gmail_users[["Name", "Email"]])

In [ ]:
# Names that start with 'K'

print(df[df["First_Name"].str.startswith("K")][["Name", "City"]])

In [ ]:
# Names that end with 'a'

print(df[df["First_Name"].str.endswith("a")][["Name", "City"]])

In [ ]:
# Check if phone number contains only digits (after cleaning)
# str.isdigit() returns True if ALL characters are digits

print(df["Phone"].str.isdigit())

`na=False` in `.str.contains()` is important — if a value is `NaN`, it returns `False` instead of throwing an error.

## 7) String Length

In [ ]:
# Length of each name

df["Name_Length"] = df["Name"].str.len()
print(df[["Name", "Name_Length"]])

In [ ]:
# Filter: phone numbers that are not 10 digits (data quality check)

invalid_phones = df[df["Phone"].str.len() != 10]
print(invalid_phones[["Name", "Phone"]])

## 8) Padding and Formatting Strings

Sometimes you need to format strings to a fixed width — like adding leading zeros to IDs or codes.

In [ ]:
# Student IDs — pad with leading zeros to always be 5 digits

ids = pd.Series(["12", "345", "7", "4521", "99"])

# zfill() pads with zeros from the left
print(ids.str.zfill(5))

In [ ]:
# pad() — more flexible, lets you choose the character and side

print(ids.str.pad(width=6, side="left", fillchar="0"))   # pad with 0 on left
print(ids.str.pad(width=6, side="right", fillchar="-"))  # pad with - on right

## 9) Putting It All Together — A Realistic Cleaning Workflow

Let's take a messy raw dataset and clean the entire text section in one go.

In [ ]:
# Raw messy data

raw = pd.DataFrame({
    "full_name": ["  john DOE ", "jane smith", "BOB MARTIN", " alice Brown"],
    "email":     ["John@Gmail.COM", "jane@yahoo.com", "BOB@outlook.com", "alice@gmail.com"],
    "phone":     ["98-765-43210", "876 543 2100", "9812345678", "  77 8899 0011"],
    "city":      ["mumbai ", "DELHI", " pune", "Mumbai"]
})

print("Before cleaning:")
print(raw)
print()

In [ ]:
# Clean it all

raw["full_name"] = raw["full_name"].str.strip().str.title()
raw["email"]     = raw["email"].str.strip().str.lower()
raw["phone"]     = raw["phone"].str.strip().str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
raw["city"]      = raw["city"].str.strip().str.title()

# Split name into first and last
raw["first_name"] = raw["full_name"].str.split(" ").str[0]
raw["last_name"]  = raw["full_name"].str.split(" ").str[1]

# Extract email domain
raw["domain"] = raw["email"].str.split("@").str[1]

# Flag invalid phone numbers
raw["phone_valid"] = raw["phone"].str.len() == 10

print("After cleaning:")
print(raw)

## Summary

All string operations use the `.str` accessor on a column:

| Operation | Method |
|---|---|
| Lowercase | `.str.lower()` |
| Uppercase | `.str.upper()` |
| Title Case | `.str.title()` |
| Remove spaces | `.str.strip()` |
| Replace characters | `.str.replace("old", "new", regex=False)` |
| Split into parts | `.str.split("separator")` |
| Get nth part after split | `.str.split("x").str[n]` |
| Slice characters | `.str[start:end]` |
| Contains pattern | `.str.contains("pattern", na=False)` |
| Starts with | `.str.startswith("x")` |
| Ends with | `.str.endswith("x")` |
| String length | `.str.len()` |
| Pad with zeros | `.str.zfill(width)` |

Next up: **Pandas Practice** — putting everything together in one project.

## Practice Questions 1

1. Create a DataFrame with 5 rows — columns: `Product_Name`, `SKU_Code`, `Category`.
   - Make `Product_Name` inconsistently cased and with extra spaces.
   - Make `SKU_Code` a mix of formats like `"sku-001"`, `"SKU 002"`, `"003"`.
2. Standardise `Product_Name` — strip spaces and apply title case.
3. Clean `SKU_Code` — remove dashes and spaces, then uppercase everything.

## Practice Questions 2

1. Filter products whose name contains the word `"pro"` (case-insensitive).
2. Split the `Product_Name` into `Brand` and `Model` columns (assume the first word is the brand).
3. Add a column `SKU_Valid` — `True` if the SKU (after cleaning) is exactly 6 characters long.